# FinWise · Risk Terminal — Colab Runner

| Field | Value |
| --- | --- |
| **Runtime** | T4 GPU (free tier, when offered) |
| **Cold start** | ≈ 8–12 min |
| **Warm click** | ≈ 20–60 s |
| **Repo** | github.com/Raghavan-27-5/complete-finwise-system |

**Do this first:** `Runtime ▸ Change runtime type ▸ T4 GPU`, then `Runtime ▸ Run all`.
Run this 15-20 minutes BEFORE the demo.

The T4 is *usually* offered on the free tier but it is **not guaranteed** (availability and
quotas vary). Every cell below also runs on a plain CPU runtime — identical output, only
slower. Full talk track, timing budget and fallback matrix: `docs/COLAB_DEMO_RUNBOOK.md`.


In [ ]:
import os

# These MUST be set BEFORE any tensorflow import anywhere in this process.
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # Keras 2 shim: the .h5 needs Keras 2
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"         # silence the TF banner / INFO noise
os.environ["PIPELINE_DEBUG"] = "0"               # FinWise pipeline debug off
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # avoid HF fork warnings / deadlocks

for _key in ("TF_USE_LEGACY_KERAS", "TF_CPP_MIN_LOG_LEVEL",
             "PIPELINE_DEBUG", "TOKENIZERS_PARALLELISM"):
    print(f"{_key} = {os.environ.get(_key)}")

print("✅ environment pinned (Keras 2 shim active)")


In [ ]:
# ===========================================================================
# CELL 3 · runtime probe + the only install this notebook needs + version audit
# ===========================================================================
import subprocess
import importlib.metadata as md


def _probe_nvidia():
    """Return `nvidia-smi -L` output, or None when there is no CUDA device."""
    try:
        proc = subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              text=True, timeout=30)
        if proc.returncode == 0 and (proc.stdout or "").strip():
            return proc.stdout.strip()
        return None
    except Exception:
        return None


_gpu = _probe_nvidia()
if _gpu:
    print("✅ GPU detected:")
    print(_gpu)
    print("→ LSTM inference and the Heston Monte-Carlo land on the T4.")
else:
    print("⚠️ no NVIDIA GPU visible (no device, or nvidia-smi is not on PATH).")
    print("→ Nothing breaks: every cell below also runs on a CPU runtime, roughly")
    print("  2-4x slower on the warm-up click. For the T4: Runtime ▸ Change runtime")
    print("  type ▸ T4 GPU, then Runtime ▸ Restart session and Run all (the env vars")
    print("  in cell 2 must be set before any tensorflow import).")
print()

# gradio is pinned to the dashboard's target version. The rest are pipeline deps.
# Everything else (tf 2.20, tf_keras 2.20.1, torch, transformers,
# sentence-transformers, spacy + en_core_web_sm, statsmodels, textblob, yfinance,
# scikit-learn) already ships in the Colab image — do NOT reinstall those.
%pip install -q "gradio==5.49.1" plotly feedparser praw PyPDF2 ta ratelimit jinja2

print()
# Version audit WITHOUT importing the heavy libraries (importlib.metadata is cheap).
_PACKAGES = (
    "gradio", "plotly", "tensorflow", "tf_keras", "torch", "transformers",
    "sentence-transformers", "spacy", "statsmodels", "textblob", "yfinance",
    "scikit-learn",
)

print(f"{'package':<24}{'version'}")
print("-" * 44)
_missing = []
for _pkg in _PACKAGES:
    try:
        _ver = md.version(_pkg)
    except Exception:
        _ver = "absent"
        _missing.append(_pkg)
    print(f"{_pkg:<24}{_ver}")

print()
if _missing:
    print("⚠️ absent:", ", ".join(_missing))
    print("   → re-run the %pip line above, then RESTART the runtime (not just the cell).")
else:
    print("✅ all expected packages visible to this interpreter")


In [ ]:
# ===========================================================================
# CELL 4 · OPTIONAL secret (NEWS_API_KEY only — never hardcoded in this file)
# Add it via the 🔑 key icon in Colab's left sidebar ▸ name it NEWS_API_KEY.
# Absent or unreadable → silently skipped: the pipeline still has key-free
# sources (Google RSS, Yahoo Finance, SEC EDGAR).
# ===========================================================================
import os

news_key = None
try:
    from google.colab import userdata
    news_key = userdata.get("NEWS_API_KEY")
except Exception:
    news_key = None

if news_key and str(news_key).strip():
    os.environ["NEWS_API_KEY"] = str(news_key).strip()
    print("✅ NEWS_API_KEY loaded from Colab secrets (the value is never printed).")
else:
    print("ℹ️ No NEWS_API_KEY → running on key-free sources "
          "(Google RSS, Yahoo Finance, SEC EDGAR)")


In [ ]:
# ===========================================================================
# CELL 5 · get the repo — clone once, then fast-forward pull on re-runs
# ===========================================================================
import pathlib
import subprocess

REPO_URL = "https://github.com/Raghavan-27-5/complete-finwise-system.git"
REPO_DIR = "/content/finwise"


def _git(args, cwd=None):
    """Run git, returning (returncode, stdout, stderr) — never raises."""
    proc = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    return proc.returncode, (proc.stdout or "").strip(), (proc.stderr or "").strip()


if pathlib.Path(REPO_DIR, ".git").exists():
    print(f"Repo already at {REPO_DIR} → fetch + fast-forward only")
    rc, out, err = _git(["fetch", "origin"], cwd=REPO_DIR)
    if rc != 0:
        print("⚠️ git fetch failed")
        print("STDOUT:", out)
        print("STDERR:", err)
    rc, out, err = _git(["pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)
    if rc != 0:
        print("⚠️ git pull --ff-only failed (diverged, or local edits in the VM?)")
        print("STDOUT:", out)
        print("STDERR:", err)
    elif out:
        print(out)
else:
    print(f"Cloning {REPO_URL} → {REPO_DIR}")
    rc, out, err = _git(["clone", "--depth", "1", REPO_URL, REPO_DIR])
    if rc != 0:
        print("⚠️ git clone failed")
        print("STDOUT:", out)
        print("STDERR:", err)
    elif out:
        print(out)

_, commit, _ = _git(["rev-parse", "--short", "HEAD"], cwd=REPO_DIR)
_, branch, _ = _git(["rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_DIR)
_files = [p for p in pathlib.Path(REPO_DIR).rglob("*")
          if p.is_file() and ".git" not in p.parts]

print("-" * 44)
print(f"commit : {commit or 'unavailable'}")
print(f"branch : {branch or 'unavailable'}")
print(f"files  : {len(_files)}")
print("(The demo code lives in the repo — Colab runtimes are ephemeral, never edit here.)")


In [ ]:
# ===========================================================================
# CELL 6 · chdir into the repo, sanity-check the artifacts, timed import
# ===========================================================================
import os
import sys
import time

os.chdir("/content/finwise")
if "/content/finwise" not in sys.path:
    sys.path.insert(0, "/content/finwise")

assert os.path.exists("intelligence/price/stock_price_model.h5"), (
    "Missing intelligence/price/stock_price_model.h5 — the trained LSTM checkpoint. "
    "You are probably not in the repo root: re-run cell 5 and read its output "
    "(truncated shallow clone? network error?)."
)
assert os.path.exists("app/gradio_app.py"), (
    "Missing app/gradio_app.py — the clone did not land in /content/finwise. "
    "Re-run cell 5 and read its stderr."
)
print("✅ repo root sane: LSTM checkpoint + dashboard entrypoint both present")

_started = time.perf_counter()
import app.gradio_app as dash
print(f"✅ imported app.gradio_app in {time.perf_counter() - _started:.1f}s")

from core.symbols import normalize_symbol, resolve_symbol

print(f"callback   : dash.run_finwise.__name__ = {dash.run_finwise.__name__}")
print(f"resolver   : from core.symbols import resolve_symbol "
      f"(module={resolve_symbol.__module__})")
print(f"normalizer : from core.symbols import normalize_symbol "
      f"(module={normalize_symbol.__module__})")
print(f"ui object  : dash.demo is a {type(dash.demo).__name__} (importable, not launched yet)")


In [ ]:
# ===========================================================================
# CELL 7 · OFFLINE UI SELF-TEST — no network, no real models.
#
# dash.compute_state is monkeypatched with a synthetic state factory, then the
# REAL dash.run_finwise("AAPL", 7) is called and the frozen 8-output contract
# from docs/DEMO_CONTRACT.md section 3 is asserted. The original is restored
# afterwards no matter what happens.
#
# Markers asserted below are REAL strings grepped out of app/gradio_app.py:
#   KPI deck       'class="kpi-grid"'                      (_kpi_html)
#   KPI symbol     '<span class="kpi-chip">symbol AAPL</span>'   (_kpi_html)
#   status strip   'class="status-strip"'                  (_status_html)
#   aspects panel  'class="aspect-panel"'                  (_aspects_html)
#   aspect label   key "supply_chain" renders as "Supply Chain"
#                  (str(name).replace("_", " ").title(), _aspects_html)
#   articles table 'class="art-table"'                     (_articles_html)
#   error banner   'class="error-banner"' — must be ABSENT (out[7] == "")
# ===========================================================================
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import core.symbols as _symbols


def _mc_figure(points=7, paths=5, seed=7):
    """Synthetic Monte-Carlo fan: 5 unnamed sample paths + 3 named envelopes."""
    rng = np.random.default_rng(seed)
    x = list(range(points))
    fig = go.Figure()
    for _ in range(paths):
        walk = 100.0 + np.cumsum(rng.normal(0.0, 0.9, size=points))
        fig.add_trace(go.Scatter(x=x, y=walk, mode="lines"))   # unnamed fan paths
    mean = 100.0 + 0.35 * np.arange(points)
    fig.add_trace(go.Scatter(x=x, y=mean + 6.0, mode="lines", name="Upper 97.5%"))
    fig.add_trace(go.Scatter(x=x, y=mean - 6.0, mode="lines", name="Lower 2.5%"))
    fig.add_trace(go.Scatter(x=x, y=mean, mode="lines", name="Mean Forecast"))
    fig.update_layout(title=dict(text="synthetic Monte-Carlo risk envelope"))
    return fig


def _synthetic_state(symbol="AAPL", days=7):
    """A complete FinWise state: 240-pt history, 7-pt forecast, MC fan, 2 aspects."""
    rng = np.random.default_rng(42)
    steps = np.arange(240)
    actual = 100.0 + 4.0 * np.sin(steps / 18.0) + np.cumsum(rng.normal(0.0, 0.35, 240))
    predicted = actual + rng.normal(0.0, 0.6, 240)
    hist_dates = pd.date_range("2024-01-02", periods=240, freq="B").strftime("%Y-%m-%d").tolist()
    fwd_dates = pd.date_range("2025-01-02", periods=days, freq="B").strftime("%Y-%m-%d").tolist()
    last = float(actual[-1])
    return {
        "symbol": symbol,
        "as_of": "2025-01-02T00:00:00+00:00",
        "price": {"trend": "Bullish", "forecast_slope": 0.42, "downside_risk": -0.031},
        "sentiment": {
            "global_score": 0.42,
            "label": "Positive",
            "aspects": {
                "supply_chain": {"label": "Supply Chain", "score": 0.42},
                "regulatory_pressure": {"label": "Regulatory Pressure", "score": -0.22},
            },
        },
        "history": {"dates": hist_dates,
                    "actual": actual.tolist(),
                    "predicted": predicted.tolist()},
        "forecast": {"dates": fwd_dates,
                     "predicted": [last + 0.4 * (i + 1) for i in range(days)]},
        "monte_carlo": {"downside_var": 0.0523, "figure": _mc_figure(days)},
        "indicators": {"rsi": 58.2, "macd": 0.31, "signal": 0.22, "momentum": 0.11},
        "articles": [{
            "headline": "Synthetic headline: AAPL services margin expansion",
            "source": "SelfTest",
            "published": "2025-01-01T12:00:00+00:00",
            "impact": 0.37, "magnitude": 0.55, "weight": 0.80,
            "direction": 1.0, "entities": ["Apple Inc", "Services"],
        }],
    }


# run_finwise() also calls resolve_symbol("AAPL"), which normally probes Yahoo.
# Pre-seed the module-level memo dict core.symbols keeps (_PROBE_CACHE) so this
# self-test really is offline. Pipeline behaviour is otherwise unchanged.
_probe_cache = getattr(_symbols, "_PROBE_CACHE", None)
if isinstance(_probe_cache, dict):
    _probe_cache.setdefault("AAPL", True)
    print("ℹ️ pre-seeded core.symbols._PROBE_CACHE['AAPL'] = True (keeps the self-test offline)")

_original_compute_state = dash.compute_state
dash.compute_state = lambda symbol, days, kg_writer=None: _synthetic_state(symbol, days)
try:
    _t0 = time.perf_counter()
    out = dash.run_finwise("AAPL", 7)          # REAL callback, synthetic state
    _selftest_seconds = time.perf_counter() - _t0
finally:
    dash.compute_state = _original_compute_state   # ALWAYS restore
print(f"↩ restored dash.compute_state → {dash.compute_state.__module__}."
      f"{dash.compute_state.__name__}")

# ---- frozen-contract assertions -------------------------------------------
_failures = []


def _check(label, ok, detail=""):
    if ok:
        print(f"  ✔ {label}")
    else:
        print(f"  ✘ {label} {detail}")
        _failures.append(label)


_check("len(out) == 8", len(out) == 8, f"-> got {len(out)}")
_check("out[7] == '' (no error banner)", out[7] == "", f"-> {out[7]!r}"[:240])
for _i, _name in ((0, "kpi_html"), (1, "status_html"),
                  (5, "aspects_html"), (6, "articles_html")):
    _check(f"out[{_i}] {_name} is a non-empty str",
           isinstance(out[_i], str) and len(out[_i]) > 0)
for _i, _name in ((2, "hist_plot"), (3, "mc_plot"), (4, "fc_plot")):
    _check(f"out[{_i}] {_name} is not None", out[_i] is not None)
_check('KPI deck marker: class="kpi-grid"', 'class="kpi-grid"' in out[0])
_check("KPI shows the resolved symbol ('symbol AAPL')", "symbol AAPL" in out[0])
_check('status strip marker: class="status-strip"', 'class="status-strip"' in out[1])
_check('aspects panel marker: class="aspect-panel"', 'class="aspect-panel"' in out[5])
_check('aspect label text rendered: "Supply Chain"', "Supply Chain" in out[5])
_check('articles table marker: class="art-table"', 'class="art-table"' in out[6])
_check('no error banner anywhere', 'class="error-banner"' not in (out[0] + out[6] + out[7]))

assert not _failures, "UI self-test failures: " + ", ".join(_failures)
print(f"✅ UI self-test passed in {_selftest_seconds:.2f}s (8/8 outputs)")

# ---- resolver assertions that need NO network ------------------------------
# resolve_symbol("AAPL") DOES probe Yahoo, so bare-ticker resolution is only
# exercised above through the pre-seeded cache. The cases below are pure
# pass-throughs (index prefix / explicit exchange suffix) and never probe.
assert normalize_symbol(" reliance ") == "RELIANCE", "normalize_symbol must strip + upper"
assert resolve_symbol("^NSEI")[0] == "^NSEI", "index symbols must pass through untouched"
assert resolve_symbol("^NSEI")[1] == ["^NSEI"], "index probe chain must be ['^NSEI']"
assert resolve_symbol("TCS.NS")[0] == "TCS.NS", "explicit .NS must pass through untouched"
assert resolve_symbol("TCS.NS")[1] == ["TCS.NS"], ".NS probe chain must be ['TCS.NS']"
print("✅ resolver offline pass-throughs OK: ' reliance ' → RELIANCE · ^NSEI · TCS.NS")


In [ ]:
# ===========================================================================
# CELL 8 · REAL WARM-UP — heavy: live network + real models (this is the slow one)
#
# The FIRST heavy cell in a session downloads FinBERT (~440 MB) and MiniLM and
# loads the LSTM + spaCy: budget 3-8 minutes. Everything after this is warm.
# ===========================================================================
import time


def _dig(mapping, *keys, default=None):
    """Safe nested read: any missing/None link returns `default`."""
    current = mapping
    for key in keys:
        if not isinstance(current, dict):
            return default
        current = current.get(key)
    return default if current is None else current


print("⏳ warm-up 1/2 · dash.compute_state('AAPL', 7, None) — live network, real models")
print("   (first run downloads FinBERT + MiniLM ~500 MB into the Colab cache)")
_t0 = time.perf_counter()
try:
    state = dash.compute_state("AAPL", 7, None)
    _warm_seconds = time.perf_counter() - _t0

    print()
    print(f"elapsed      : {_warm_seconds:.1f}s")
    print(f"trend        : {_dig(state, 'price', 'trend')}")
    print(f"DSP          : {_dig(state, 'sentiment', 'global_score')}")
    print(f"articles     : {len(_dig(state, 'articles', default=[]) or [])}")
    print(f"5% VaR       : {_dig(state, 'monte_carlo', 'downside_var')}")
    print(f"aspects      : {len(_dig(state, 'sentiment', 'aspects', default={}) or {})}")
except Exception as exc:
    print(f"⚠️ warm-up failed: {type(exc).__name__}: {exc}")
    print("   → the self-test still proves the UI; check network/DNS on the runtime,")
    print("     then re-run this cell. See the troubleshooting matrix in cell 12.")

# Flip to False to skip the second (cheap) warm-up and go straight to launch.
WARM_SECOND_ASSET = True
SECOND_ASSETS = ["RELIANCE.NS"]      # extend this list for extra Indian tickers

if WARM_SECOND_ASSET:
    for SECOND_ASSET in SECOND_ASSETS:
        print()
        print(f"⏳ warm-up 2/2 · dash.compute_state('{SECOND_ASSET}', 7, None) — Indian ticker")
        _t0 = time.perf_counter()
        try:
            state2 = dash.compute_state(SECOND_ASSET, 7, None)
            print(f"elapsed      : {time.perf_counter() - _t0:.1f}s")
            print(f"trend        : {_dig(state2, 'price', 'trend')}")
            print(f"articles     : {len(_dig(state2, 'articles', default=[]) or [])}")
            print(f"5% VaR       : {_dig(state2, 'monte_carlo', 'downside_var')}")
        except Exception as exc:
            print(f"   ⚠️ {SECOND_ASSET} warm-up failed (non-fatal): "
                  f"{type(exc).__name__}: {exc}")
            print("   → type the explicit .NS suffix in the UI; price/risk still work.")

print()
print("✅ pipeline warm — live clicks are now fast (expect ~20-60 s for a fresh ticker).")
print("ℹ️ Reminder: the FIRST heavy run in a fresh session downloads FinBERT/MiniLM")
print("   (~500 MB). Colab caches them for the session, so cell 9 is safe to run after.")


In [ ]:
# ===========================================================================
# CELL 9 · LAUNCH — the real demo surface
#
# share=True prints a public https://*.gradio.live URL (open THAT in a new tab)
# and, on Colab, usually also renders the app inline below as an iframe.
# KEEP THIS CELL RUNNING — evaluating it again while a job is in flight only
# stacks requests. Stop it with the ■ square when you are done.
# ===========================================================================
import app.gradio_app as dash   # no-op if cell 6 already imported it

dash.demo.queue(default_concurrency_limit=2)
dash.demo.launch(share=True, show_error=True, quiet=False)


In [ ]:
# ===========================================================================
# CELL 10 · PLAN B — run ONLY if cell 9 failed
#
# Symptoms that send you here: "Could not create share link", share=True
# returning no URL, gradio.live blocked by the network, or the tunnel dying
# mid-demo. This path does NOT need a public tunnel: it serves the same app on
# port 7860 inside the Colab VM and Colab proxies that port to a signed URL
# (a *.googleusercontent.com link that interpreters can also open).
# ===========================================================================
from google.colab import output

output.serve_kernel_port_as_window(7860)
dash.demo.queue(default_concurrency_limit=2)
dash.demo.launch(share=False, server_name="0.0.0.0", server_port=7860,
                 show_error=True)


In [ ]:
# ===========================================================================
# CELL 11 · KEEP-ALIVE — leave this tab OPEN and this cell RUNNING.
#
# Colab disconnects idle runtimes (and free-tier sessions are recycled after
# roughly 12 h). This clicks the "connect" button once a minute so the session
# is treated as active. Harmless if you are already clicking in the UI; stop it
# with the ■ square.
# ===========================================================================
from IPython.display import Javascript, display

display(Javascript("""
setInterval(() => {
  const b = document.querySelector("colab-connect-button");
  if (b) b.click();
}, 60000);
"""))
print("⏱️ keep-alive armed — one synthetic connect click every 60 s.")
print("   Do NOT close this browser tab and do NOT stop this cell during the demo.")


## Troubleshooting matrix

| Symptom | Cause | Fix |
| --- | --- | --- |
| `share=True` never returns a URL | The gradio.live tunnel could not be established (network policy, proxy, or the tunnel endpoint is down) | Stop cell 9 and run **cell 10 (Plan B)** — it serves port 7860 inside the VM and Colab proxies it to a signed URL |
| `ValueError: Unknown layer` / `could not deserialize` while loading the LSTM | TensorFlow picked Keras 3, but `stock_price_model.h5` was saved with Keras 2 | `Runtime ▸ Restart session`, then `Runtime ▸ Run all`. **Cell 2 must run before any tensorflow import** — `TF_USE_LEGACY_KERAS=1` is what selects the Keras 2 shim |
| `ModuleNotFoundError: gradio` (or wrong version) | Cell 3 was skipped after a runtime restart, or pip installed into a different interpreter | Re-run **cell 3**, then `Runtime ▸ Restart session` and `Runtime ▸ Run all` (pip installs need a fresh interpreter) |
| `yfinance` returns empty for an Indian ticker | Bare `RELIANCE` resolved to a US listing/ADR, or the suffix probe was rate-limited | Type the explicit ticker: **`RELIANCE.NS`** (or `.BO`). Index symbols (`^NSEI`) and explicit suffixes pass straight through with no probe |
| Sentiment panel shows the empty state | No relevant articles inside the window (thin coverage or a fresh holiday period) | Expected and non-fatal: price, forecast and the Monte-Carlo risk envelope still render. Pick a more covered ticker (AAPL, NVDA) for the demo |
| `Runtime disconnected` / session recycled | Colab killed an idle session, or the tab was closed | Keep the tab open and **cell 11 running**, do one exploratory click before the demo, and re-run cells 2 ▸ 6 (you do not need to reinstall if the runtime survived) |
| Very slow first click (minutes, not seconds) | Cold process: FinBERT/MiniLM weights are not in the page cache yet, and the LSTM/spaCy have not been loaded | Run **cell 8** warm-up (it does exactly one `compute_state` call) *before* the demo. Per-click latency then drops to ≈20-60 s |

### The T4 reality

Colab's free tier *usually* offers a T4 GPU, but it is **not guaranteed** — availability and
weekly quotas vary by region and time of day. Nothing in this notebook depends on the GPU:
a CPU runtime produces identical numbers, just slower (roughly 2-4x on the warm-up call).
If the T4 is unavailable, run the demo on CPU and say so — the architecture story is unchanged.
